# 07 - Phân tích ván đấu (agent tìm kiếm: Heuristic, Alpha-Beta, MCTS)

Notebook này CHỈ đọc log thô `results/logs/<RUN_NAME>/games.jsonl` và gọi `eval/game_analysis.py` (toàn bộ logic nằm trong file .py để test và chạy lại bằng 1 lệnh). Không chạy lại agent.

Nguyên tắc theo đề: mọi con số tính trên **toàn bộ ván**; ván minh họa chọn bằng **quy tắc viết sẵn** (ván thua nặng nhất, sát nút nhất, dài nhất, ngẫu nhiên có seed), không chọn riêng ván thắng.

**Chỉ điền phần nhận xét sau khi chạy với log chính thức.** Log tune (`tune_*`) chỉ dùng để chọn tham số, không đưa vào kết luận.

In [ ]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "eval" / "game_analysis.py").exists())
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, Markdown, display
from eval.game_analysis import analyze, QUY_TAC_CHON_VAN

RUN_NAME = "round_robin_v1"          # đổi thành tên lần chạy chính thức (đã chạy xong, đã đóng băng agent)
FOCUS = ["alphabeta", "mcts"]       # cặp cần chọn ván minh họa; None = mọi cặp
RUN_DIR = ROOT / "results" / "logs" / RUN_NAME
OUT_DIR = ROOT / "results" / "analysis" / RUN_NAME
pd.set_option("display.width", 200, "display.max_columns", 40)
R = analyze(RUN_DIR, OUT_DIR, FOCUS)
print(R["n_games"], "ván đã phân tích; kết quả ghi vào", OUT_DIR)

## 1. Ai thắng ai, theo từng chế độ ngân sách
`a_score` = (thắng + 0.5 hòa) / số ván của agent `a`; CI 95% Wilson. `pairs_*` đếm theo CẶP ván cùng khai cuộc (đổi màu) nên loại được may rủi của khai cuộc; `black_win_rate` cho thấy lợi thế đi trước.

In [ ]:
R['summary']

In [ ]:
display(Image(str(OUT_DIR / 'fig_matchup_scores.png')))

## 2. Chi phí tính toán theo giai đoạn ván
Nút (Alpha-Beta/Heuristic) hoặc mô phỏng (MCTS) mỗi nước, độ sâu đạt được, thời gian trung bình và p95, tỷ lệ vượt ngân sách.

In [ ]:
R['move_stats']

In [ ]:
display(Image(str(OUT_DIR / 'fig_cost_by_phase.png')))

## 3. Hành vi chiến thuật: góc, ô X/C
`gave_corner_rate` = tỷ lệ nước đi MỞ cho đối thủ một góc mà trước đó họ chưa lấy được (dấu hiệu horizon effect hoặc hàm đánh giá thiếu tầm nhìn).

In [ ]:
display(Image(str(OUT_DIR / 'fig_corner_behaviour.png')))

In [ ]:
R['corner_vs_result']

## 4. Quỹ đạo chênh lệch quân (gồm cả ván thua)
Mỗi đường là trung bình theo nước đi của các ván cùng kết quả; xem agent bắt đầu tụt từ giai đoạn nào.

In [ ]:
for p in sorted(OUT_DIR.glob('fig_trajectory_*.png')):
    display(Markdown('**' + p.stem + '**')); display(Image(str(p)))

## 5. MCTS ước lượng thắng có đáng tin không
So giá trị MCTS tự ước lượng ở mỗi nước với kết quả thật của ván (Brier càng thấp càng tốt).

In [ ]:
cal = R['calibration']
display(cal if len(cal) else 'Log không có stats.value của MCTS')
f = OUT_DIR / 'fig_mcts_calibration.png'
if f.exists(): display(Image(str(f)))

## 6. Ván minh họa (chọn theo quy tắc cố định)

In [ ]:
display(Markdown(QUY_TAC_CHON_VAN))
R['selected']

Bàn cờ tại nước 20/40/cuối của từng ván nằm trong `selected_games.md` (cùng thư mục kết quả).

## 7. Nhận xét (điền SAU KHI chạy log chính thức, mỗi ý kèm số liệu từ bảng trên)

- **Câu hỏi 1** MCTS vs Alpha-Beta cùng thời gian: kết quả ở từng chế độ ngân sách, CI có chứa 0.5 không?
- Alpha-Beta đạt độ sâu trung bình bao nhiêu trong 0.5 s? MCTS chạy bao nhiêu mô phỏng? Giai đoạn nào cả hai tốn nhiều nhất?
- **Câu hỏi 2** góc và mobility: người lấy góc đầu tiên / bên hơn mobility ở nước 20 thắng bao nhiêu phần trăm?
- Lỗi điển hình: `gave_corner_rate` của agent nào cao, có tập trung ở giai đoạn nào?
- Giới hạn: số ván, số seed, MCTS rollout ngẫu nhiên, một máy một cấu hình.